# 00 Quickstart

- legge `../dataset.yml`
- mostra i path reali attesi dal toolkit
- esegue il run solo se abiliti esplicitamente `RUN_TOOLKIT = True`

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import yaml

ROOT = Path('.').resolve()
DATASET_YML = (ROOT / 'dataset.yml').resolve() if (ROOT / 'dataset.yml').exists() else (ROOT / '..' / 'dataset.yml').resolve()
CFG = yaml.safe_load(DATASET_YML.read_text(encoding='utf-8'))
BASE_DIR = DATASET_YML.parent
DATASET = CFG['dataset']['name']
YEARS = CFG['dataset']['years']
YEAR_INDEX = 0
YEAR = YEARS[YEAR_INDEX] if YEARS and 0 <= YEAR_INDEX < len(YEARS) else YEARS[0]
MART_TABLES = [table['name'] for table in CFG.get('mart', {}).get('tables', [])]
RUN_TOOLKIT = False
CLI_PREFIX = ['toolkit'] if shutil.which('toolkit') else ['py', '-m', 'toolkit.cli.app']
RUN_CMD = CLI_PREFIX + ['run', 'all', '--config', str(DATASET_YML)]
VALIDATE_CMD = CLI_PREFIX + ['validate', 'all', '--config', str(DATASET_YML)]
INSPECT_CMD = CLI_PREFIX + ['inspect', 'paths', '--config', str(DATASET_YML), '--year', str(YEAR), '--json']
try:
    INSPECT = json.loads(subprocess.run(INSPECT_CMD, capture_output=True, text=True, check=True).stdout)
except Exception:
    OUT_ROOT = (BASE_DIR / CFG.get('root', '.')).resolve()
    INSPECT = {
        'root': str(OUT_ROOT),
        'paths': {
            'raw': {'dir': str(OUT_ROOT / 'data' / 'raw' / DATASET / str(YEAR))},
            'clean': {'dir': str(OUT_ROOT / 'data' / 'clean' / DATASET / str(YEAR))},
            'mart': {'dir': str(OUT_ROOT / 'data' / 'mart' / DATASET / str(YEAR))},
            'run_dir': str(OUT_ROOT / 'data' / '_runs' / DATASET / str(YEAR)),
        },
        'latest_run': None,
    }

{
    'DATASET_YML': str(DATASET_YML),
    'ROOT': INSPECT.get('root'),
    'DATASET': DATASET,
    'YEARS': YEARS,
    'YEAR_INDEX': YEAR_INDEX,
    'YEAR': YEAR,
    'MART_TABLES': MART_TABLES,
    'CLI_PREFIX': CLI_PREFIX,
    'INSPECT_CMD': INSPECT_CMD,
}

In [ ]:
{
    'raw_dir': INSPECT['paths']['raw']['dir'],
    'clean_dir': INSPECT['paths']['clean']['dir'],
    'mart_dir': INSPECT['paths']['mart']['dir'],
    'run_dir': INSPECT['paths']['run_dir'],
    'latest_run': INSPECT.get('latest_run'),
}

In [ ]:
print('Run command:', ' '.join(RUN_CMD))
print('Validate command:', ' '.join(VALIDATE_CMD))

if RUN_TOOLKIT:
    subprocess.run(RUN_CMD, check=True)
    subprocess.run(VALIDATE_CMD, check=True)
else:
    print('Toolkit run disabled. Set RUN_TOOLKIT = True to execute the pipeline.')